In [1]:
import json
import numpy as np

SEEDS = [42, 142, 242]

all_configs = [
    # Phase 1 — sweep T
    {"timesteps": 100, "base_channels": 32, "lr": 1e-4},
    {"timesteps": 200, "base_channels": 32, "lr": 1e-4},
    {"timesteps": 400, "base_channels": 32, "lr": 1e-4},
    # Phase 2 — sweep ch (T=200)
    {"timesteps": 200, "base_channels": 16, "lr": 1e-4},
    {"timesteps": 200, "base_channels": 64, "lr": 1e-4},
    # Phase 3 — sweep lr (T=200, ch=64)
    {"timesteps": 200, "base_channels": 64, "lr": 1e-5},
    {"timesteps": 200, "base_channels": 64, "lr": 1e-3},
]

print("=== ALL RESULTS ===\n")
results = {}

for cfg in all_configs:
    cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"
    fid_scores = []
    for seed in SEEDS:
        path = f"ddpm_{cfg_name}/seed_{seed}/history.json"
        with open(path) as f:
            history = json.load(f)
        fid_scores.append(history["fid"][-1]["fid"])

    mean = np.mean(fid_scores)
    std = np.std(fid_scores)
    results[cfg_name] = {"mean": mean, "std": std, "cfg": cfg}
    print(f"{cfg_name}: FID = {mean:.2f} ± {std:.2f}  (seeds: {[f'{f:.1f}' for f in fid_scores]})")

best_name = min(results, key=lambda k: results[k]["mean"])
best_cfg = results[best_name]["cfg"]
print(f"\nBest overall: {best_name}  (FID = {results[best_name]['mean']:.2f} ± {results[best_name]['std']:.2f})")
print(f"=> Finalny config: T={best_cfg['timesteps']}, ch={best_cfg['base_channels']}, lr={best_cfg['lr']}")

=== ALL RESULTS ===

T100_ch32_lr0.0001: FID = 154.49 ± 4.95  (seeds: ['157.7', '147.5', '158.3'])
T200_ch32_lr0.0001: FID = 148.53 ± 10.04  (seeds: ['159.2', '151.4', '135.1'])
T400_ch32_lr0.0001: FID = 162.78 ± 9.70  (seeds: ['170.7', '149.1', '168.6'])
T200_ch16_lr0.0001: FID = 296.55 ± 11.71  (seeds: ['281.6', '297.8', '310.2'])
T200_ch64_lr0.0001: FID = 81.64 ± 8.50  (seeds: ['87.2', '69.6', '88.1'])
T200_ch64_lr1e-05: FID = 199.45 ± 6.18  (seeds: ['201.3', '191.1', '205.9'])
T200_ch64_lr0.001: FID = 142.09 ± 18.26  (seeds: ['129.0', '129.3', '167.9'])

Best overall: T200_ch64_lr0.0001  (FID = 81.64 ± 8.50)
=> Finalny config: T=200, ch=64, lr=0.0001
